In [32]:
from pathlib import Path
import pandas as pd

ESCO_DIR = Path(r"C:\Users\DELL\OneDrive\Máy tính\job-market-radar\data\reference\esco")  # <-- point to the unzipped folder

csv_files = sorted(ESCO_DIR.rglob("*.csv"))
len(csv_files), [p.name for p in csv_files[:30]]


(18,
 ['broaderRelationsOccPillar_de.csv',
  'broaderRelationsSkillPillar_de.csv',
  'dictionary_de.csv',
  'digCompSkillsCollection_de.csv',
  'digitalSkillsCollection_de.csv',
  'greenShareOcc_de.csv',
  'greenSkillsCollection_de.csv',
  'ISCOGroups_de.csv',
  'languageSkillsCollection_de.csv',
  'occupations_de.csv',
  'occupationSkillRelations_de.csv',
  'researchOccupationsCollection_de.csv',
  'researchSkillsCollection_de.csv',
  'skillGroups_de.csv',
  'skills_de.csv',
  'skillsHierarchy_de.csv',
  'skillSkillRelations_de.csv',
  'transversalSkillsCollection_de.csv'])

In [42]:
from pathlib import Path
import pandas as pd

ESCO_DIR = Path(r"C:\Users\DELL\OneDrive\Máy tính\job-market-radar\data\reference\esco")

skills_path = ESCO_DIR / "skills_de.csv"
dict_path   = ESCO_DIR / "dictionary_de.csv"

def read_esco_csv(path: Path) -> pd.DataFrame:
    # robust delimiter detection (no schema guessing)
    for sep in [",", ";", "\t", "|"]:
        try:
            df = pd.read_csv(path, sep=sep, encoding="utf-8", low_memory=False)
            if df.shape[1] > 1:
                df["_sep_used"] = sep
                return df
        except Exception:
            pass
    raise ValueError(f"Could not read {path.name} with common separators")

skills_df = read_esco_csv(skills_path)
dict_df   = read_esco_csv(dict_path)

(skills_path.name, skills_df["_sep_used"].iloc[0], skills_df.shape, skills_df.columns.tolist(), skills_df.head(3))


('skills_de.csv',
 ',',
 (13960, 14),
 ['conceptType',
  'conceptUri',
  'skillType',
  'reuseLevel',
  'preferredLabel',
  'altLabels',
  'hiddenLabels',
  'status',
  'modifiedDate',
  'scopeNote',
  'definition',
  'inScheme',
  'description',
  '_sep_used'],
                 conceptType  \
 0  KnowledgeSkillCompetence   
 1  KnowledgeSkillCompetence   
 2  KnowledgeSkillCompetence   
 
                                           conceptUri         skillType  \
 0  http://data.europa.eu/esco/skill/0005c151-5b5a...  skill/competence   
 1  http://data.europa.eu/esco/skill/00064735-8fad...  skill/competence   
 2  http://data.europa.eu/esco/skill/000709ed-2be5...  skill/competence   
 
             reuseLevel                           preferredLabel altLabels  \
 0      sector-specific                  Musikpersonal verwalten       NaN   
 1  occupation-specific    Strafvollzugsverfahren beaufsichtigen       NaN   
 2      sector-specific  nicht unterdrückende Praktiken anwenden       

In [43]:
(dict_path.name, dict_df["_sep_used"].iloc[0], dict_df.shape, dict_df.columns.tolist(), dict_df.head(3))


('dictionary_de.csv',
 ',',
 (153, 5),
 ['filename', 'data header', 'property', 'description', '_sep_used'],
       filename  data header                                      property  \
 0  occupations  conceptType                                           NaN   
 1  occupations   conceptUri                                           NaN   
 2  occupations    iscoGroup  http://www.w3.org/2004/02/skos/core#notation   
 
                                          description _sep_used  
 0                  No direct related property exists         ,  
 1                  No direct related property exists         ,  
 2  A notation, also known as classification code,...         ,  )

In [44]:
import numpy as np

TARGET_TERMS = [
    "python", "sql", "power bi", "azure", "databricks",
    "vertex ai", "copilot studio", "azure machine learning", "azure ml",
    "google vertex", "llm", "machine learning"
]

def term_hits(df: pd.DataFrame, terms):
    # search across ALL string columns (no schema assumptions)
    obj = df.select_dtypes(include=["object"]).copy()
    # drop helper column
    if "_sep_used" in obj.columns:
        obj = obj.drop(columns=["_sep_used"])
    text = obj.fillna("").astype(str)

    # any column contains term
    out = []
    for t in terms:
        m = text.apply(lambda s: s.str.contains(t, case=False, regex=False)).any(axis=1)
        out.append({"term": t, "hits": int(m.sum())})
    return pd.DataFrame(out).sort_values("hits", ascending=False)

hits_skills = term_hits(skills_df, TARGET_TERMS)
hits_dict   = term_hits(dict_df, TARGET_TERMS)

hits_skills, hits_dict


(                      term  hits
 10                     llm    56
 1                      sql    12
 0                   python     2
 2                 power bi     0
 3                    azure     0
 4               databricks     0
 6           copilot studio     0
 5                vertex ai     0
 7   azure machine learning     0
 8                 azure ml     0
 9            google vertex     0
 11        machine learning     0,
                       term  hits
 0                   python     0
 1                      sql     0
 2                 power bi     0
 3                    azure     0
 4               databricks     0
 5                vertex ai     0
 6           copilot studio     0
 7   azure machine learning     0
 8                 azure ml     0
 9            google vertex     0
 10                     llm     0
 11        machine learning     0)

In [46]:
import pandas as pd

t = "llm"
obj = skills_df.select_dtypes(include=["object"]).drop(columns=["_sep_used"], errors="ignore").fillna("").astype(str)
mask = obj.apply(lambda s: s.str.contains(t, case=False, regex=False)).any(axis=1)

matched = skills_df.loc[mask].head(5).copy()

# which columns matched (for those first 5)
cols_hit = obj.loc[mask].head(5).apply(lambda r: [c for c,v in r.items() if t.lower() in v.lower()], axis=1)

matched, cols_hit


(                   conceptType  \
 141   KnowledgeSkillCompetence   
 174   KnowledgeSkillCompetence   
 623   KnowledgeSkillCompetence   
 993   KnowledgeSkillCompetence   
 1303  KnowledgeSkillCompetence   
 
                                              conceptUri         skillType  \
 141   http://data.europa.eu/esco/skill/0329ab41-3212...  skill/competence   
 174   http://data.europa.eu/esco/skill/03d93b02-a461...  skill/competence   
 623   http://data.europa.eu/esco/skill/0bb4b474-da3a...         knowledge   
 993   http://data.europa.eu/esco/skill/1289bd81-8daf...         knowledge   
 1303  http://data.europa.eu/esco/skill/18508acd-a561...  skill/competence   
 
                reuseLevel                                 preferredLabel  \
 141   occupation-specific                                  Dauben biegen   
 174       sector-specific                  Forderungsanmeldung verwalten   
 623          cross-sector                               Schweißverfahren   
 993      

In [49]:
TARGET = [
    "python", "sql", "power bi", "azure", "databricks",
    "maschinelles lernen", "machine learning",
    "large language model", "großes sprachmodell", "große sprachmodelle"
]

labels = skills_df[LABEL_COLS].fillna("").astype(str)

hits = []
for t in TARGET:
    mask = labels.apply(lambda s: s.str.contains(t, case=False, regex=False)).any(axis=1)
    hits.append((t, int(mask.sum())))
hits


[('python', 1),
 ('sql', 9),
 ('power bi', 0),
 ('azure', 0),
 ('databricks', 0),
 ('maschinelles lernen', 2),
 ('machine learning', 0),
 ('large language model', 0),
 ('großes sprachmodell', 0),
 ('große sprachmodelle', 0)]

In [51]:
from pathlib import Path
import pandas as pd

ESCO_DIR = Path(r"C:\Users\DELL\OneDrive\Máy tính\job-market-radar\data\reference\esco")

p = ESCO_DIR / "digitalSkillsCollection_de.csv"
df = pd.read_csv(p, encoding="utf-8", low_memory=False)

(p.name, df.shape, df.columns.tolist(), df.head(5))


('digitalSkillsCollection_de.csv',
 (1284, 10),
 ['conceptType',
  'conceptUri',
  'preferredLabel',
  'status',
  'skillType',
  'reuseLevel',
  'altLabels',
  'description',
  'broaderConceptUri',
  'broaderConceptPT'],
                 conceptType  \
 0  KnowledgeSkillCompetence   
 1  KnowledgeSkillCompetence   
 2  KnowledgeSkillCompetence   
 3  KnowledgeSkillCompetence   
 4  KnowledgeSkillCompetence   
 
                                           conceptUri  \
 0  http://data.europa.eu/esco/skill/000f1d3d-220f...   
 1  http://data.europa.eu/esco/skill/00c04e40-35ea...   
 2  http://data.europa.eu/esco/skill/013441c1-1f13...   
 3  http://data.europa.eu/esco/skill/0189f448-179e...   
 4  http://data.europa.eu/esco/skill/01d269d9-b058...   
 
                                    preferredLabel    status         skillType  \
 0                                         Haskell  released         knowledge   
 1                       inkrementelle Entwicklung  released         knowled

In [52]:
import pandas as pd

ds = df  # your digitalSkillsCollection_de.csv dataframe from earlier

label_cols = ["preferredLabel", "altLabels"]
labels = ds[label_cols].fillna("").astype(str)

targets = ["python", "sql", "maschinelles lernen", "power bi", "azure", "databricks"]

hits = []
for t in targets:
    mask = labels.apply(lambda s: s.str.contains(t, case=False, regex=False)).any(axis=1)
    hits.append((t, int(mask.sum())))
hits


[('python', 1),
 ('sql', 7),
 ('maschinelles lernen', 2),
 ('power bi', 0),
 ('azure', 0),
 ('databricks', 0)]

In [53]:
# ds is your digitalSkillsCollection dataframe (df)
alts = ds["altLabels"].dropna().astype(str)

alts.head(20)


4             innovative Blockchain-Architekturen bauen
5                                   Netzwerkarchitektur
7     physische Konfiguration von Datenbankdateien f...
18                          Elektroanlagen konstruieren
27                                     DNS-Server | DNS
31    Signaturschemata für Blockchain | Blockchain-S...
32    Daten reduzieren | Daten vorbereiten | Datenno...
33    Prototypen für User-Experience-Lösung erstelle...
35    IT-Lösungen für Unternehmensprobleme vorschlag...
37    Software zur Immobilienverwaltung | Software f...
39                         Computergestützte Simulation
41    IKT-System administrieren | Netzwerk und Syste...
42    Social-Engineering-Angriffe testweise durchfüh...
44                               Digitaldruck | Plotter
46    Entwicklungslebenszyklus eines Systems | Leben...
48         automatische Verfahrenssteuerung durchführen
53    Fahrten empfehlen | Fahrten berechnen | Routen...
54                            agile Management-S

In [54]:
{
  "n_alt_nonnull": int(ds["altLabels"].notna().sum()),
  "contains_semicolon": float(alts.str.contains(";", regex=False).mean()),
  "contains_comma": float(alts.str.contains(",", regex=False).mean()),
  "contains_pipe": float(alts.str.contains("|", regex=False).mean()),
}


{'n_alt_nonnull': 398,
 'contains_semicolon': 0.0,
 'contains_comma': 0.0,
 'contains_pipe': 0.6708542713567839}

In [24]:
# List false negatives on TEST: gold skills not found by Method A
misses = []
for k in test_key:
    gset = gold_sets_test.get(k, set())
    pset = pred_sets_test.get(k, set())
    for skill in sorted(gset - pset):
        misses.append({"source": k[0], "job_id": k[1], "missed_skill": skill})

misses_df = pd.DataFrame(misses)
misses_df


,source,job_id,missed_skill
0,adzuna,5451953400,Python
1,adzuna,5555029866,ETL/ELT
2,adzuna,5548623172,Power BI


In [25]:
for jid in ["5451953400", "5555029866", "5548623172"]:
    row = test_jobs_full[test_jobs_full["job_id"] == jid].iloc[0]
    gold_sk = sorted(list(gold_sets_test.get(("adzuna", jid), set())))
    pred_sk = sorted(list(pred_sets_test.get(("adzuna", jid), set())))
    print("\n" + "="*120)
    print("job_id:", jid, "| title:", row["title"])
    print("GOLD:", gold_sk)
    print("PRED:", pred_sk)
    print("\nDESCRIPTION:\n")
    print(row["description"])



job_id: 5451953400 | title: Praktikum Data driven Machine Diagnosis in Python (SS26)
GOLD: ['Python']
PRED: []

DESCRIPTION:

As a family-run, high-tech company with nearly 19,000 employees at 71 locations worldwide, we are looking for forward thinkers with unconventional ideas and drive to join our team. Our company culture, which values collaboration and mutual trust, creates the ideal framework for boldly trying new things and questioning the status quo. Our technologies inspire people to develop and produce things that are currently unimaginable. Whether lasers, machine tools, EUV or electronics - TRUMPF is buildi…

job_id: 5555029866 | title: Student Data Science als Studentische Hilfskraft Data Governance / ETL‑Prozesse (w/m/d)
GOLD: ['ETL/ELT']
PRED: []

DESCRIPTION:

Nachhaltig, innovativ, digital und zukunftsorientiert – E.ON. Wir sind der Arbeitgeber, der die Energiezukunft gestaltet! Mit Leidenschaft arbeiten wir daran, eine CO2-freie, digitale Welt aufzubauen und setzen da

In [26]:
# Method A' = dictionary matching on (title + "\n" + description)
pred_sets_test_td = {}
for _, r in test_jobs_full.iterrows():
    k = (r["source"], r["job_id"])
    text_td = (r["title"] if isinstance(r["title"], str) else "") + "\n" + (r["description"] if isinstance(r["description"], str) else "")
    pred_sets_test_td[k] = extract_A(text_td)

tp = fp = fn = 0
for k in test_key:
    gset = gold_sets_test.get(k, set())
    pset = pred_sets_test_td.get(k, set())
    tp += len(gset & pset)
    fp += len(pset - gset)
    fn += len(gset - pset)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2*precision*recall/(precision+recall)) if (precision+recall) else 0.0
coverage = sum(1 for k in test_key if len(pred_sets_test_td.get(k,set()))>0) / len(test_key)

{
    "test_jobs": len(test_key),
    "tp": tp, "fp": fp, "fn": fn,
    "precision": round(precision,4),
    "recall": round(recall,4),
    "f1": round(f1,4),
    "coverage(pred>=1_skill)": round(coverage,4),
    "avg_skills_per_job_pred": round(sum(len(pred_sets_test_td.get(k,set())) for k in test_key)/len(test_key),4),
}



{'test_jobs': 27,
 'tp': 14,
 'fp': 0,
 'fn': 0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'coverage(pred>=1_skill)': 0.2593,
 'avg_skills_per_job_pred': 0.5185}

In [27]:
# Build evaluation tables for TD (title + description) on DEV and TEST

def make_text_td(df_jobs):
    return (df_jobs["title"].fillna("").astype(str) + "\n" + df_jobs["description"].fillna("").astype(str))

# DEV
dev_key = set(zip(dev_jobs["source"].astype(str), dev_jobs["job_id"].astype(str)))
dev_jobs_full = gold_jobs.copy()
dev_jobs_full["job_id"] = dev_jobs_full["job_id"].astype(str)
dev_jobs_full["source"] = dev_jobs_full["source"].astype(str)
dev_jobs_full = dev_jobs_full[dev_jobs_full.apply(lambda r: (r["source"], r["job_id"]) in dev_key, axis=1)].copy()
dev_text_td = make_text_td(dev_jobs_full)
gold_sets_dev = (
    gold_pairs.copy()
      .assign(source=lambda x: x["source"].astype(str), job_id=lambda x: x["job_id"].astype(str))
)
gold_sets_dev = gold_sets_dev[gold_sets_dev.apply(lambda r: (r["source"], r["job_id"]) in dev_key, axis=1)]
gold_sets_dev = gold_sets_dev.groupby(["source","job_id"])["skill"].apply(set).to_dict()

# TEST
test_key = set(zip(test_jobs["source"].astype(str), test_jobs["job_id"].astype(str)))
test_jobs_full = gold_jobs.copy()
test_jobs_full["job_id"] = test_jobs_full["job_id"].astype(str)
test_jobs_full["source"] = test_jobs_full["source"].astype(str)
test_jobs_full = test_jobs_full[test_jobs_full.apply(lambda r: (r["source"], r["job_id"]) in test_key, axis=1)].copy()
test_text_td = make_text_td(test_jobs_full)
gold_sets_test = (
    gold_pairs.copy()
      .assign(source=lambda x: x["source"].astype(str), job_id=lambda x: x["job_id"].astype(str))
)
gold_sets_test = gold_sets_test[gold_sets_test.apply(lambda r: (r["source"], r["job_id"]) in test_key, axis=1)]
gold_sets_test = gold_sets_test.groupby(["source","job_id"])["skill"].apply(set).to_dict()

{
  "dev_jobs": len(dev_jobs_full),
  "test_jobs": len(test_jobs_full),
  "dev_jobs_with_gold": len(gold_sets_dev),
  "test_jobs_with_gold": len(gold_sets_test),
  "dev_text_len_avg": float(dev_text_td.str.len().mean()),
  "test_text_len_avg": float(test_text_td.str.len().mean()),
}


{'dev_jobs': 63,
 'test_jobs': 27,
 'dev_jobs_with_gold': 13,
 'test_jobs_with_gold': 7,
 'dev_text_len_avg': 552.5873015873016,
 'test_text_len_avg': 554.925925925926}

In [29]:
def score_sets(keys, gold_dict, pred_dict):
    tp = fp = fn = 0
    for k in keys:
        g = gold_dict.get(k, set())
        p = pred_dict.get(k, set())
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2*precision*recall/(precision+recall)) if (precision+recall) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

{"scorer_defined": True}


{'scorer_defined': True}